# Capstone — FCFS vs LTR vs PARS+ProD-M+Priority

Labels are generated in **4 checkpoints of 25 prompts** and copied to Drive after each chunk, so a disconnect does not wipe everything.

1. Runtime → GPU (T4)
2. Accept: https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct
3. Paste HF token below
4. Run cells in order

In [ ]:
import os
from google.colab import drive

os.environ["HF_TOKEN"] = "hf_YOUR_TOKEN_HERE"  # <-- paste token
drive.mount("/content/drive")

BACKUP = "/content/drive/MyDrive/capstone_results"
os.makedirs(BACKUP, exist_ok=True)

import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import shutil, os

if os.path.exists("/content/pairwise-ltr-scheduler"):
    shutil.rmtree("/content/pairwise-ltr-scheduler")

!git clone https://github.com/anmolsaluja/pairwise-ltr-scheduler.git
%cd /content/pairwise-ltr-scheduler

# IMPORTANT: do not pip-install torch from PyPI on Colab (breaks CUDA).
# Only install the other deps. If CUDA is broken, repair torch first:
import torch
print("cuda before install:", torch.cuda.is_available(), torch.__version__)
if not torch.cuda.is_available():
    !pip install -q --upgrade torch --index-url https://download.pytorch.org/whl/cu121

!pip install -q -r requirements.txt

# restore previous checkpoint from Drive if present
BACKUP = "/content/drive/MyDrive/capstone_results"
os.makedirs("data/processed", exist_ok=True)
for name in ("prod_labels.json", "prod_hidden.pt"):
    src = f"{BACKUP}/{name}"
    if os.path.exists(src):
        shutil.copy2(src, f"data/processed/{name}")
        print("restored", name)

!python scripts/check_setup.py

In [ ]:
# Label generation: 100 prompts, save every 25 (~30-60 min per chunk on T4)
# Safe to re-run: --resume skips prompts already saved
!python scripts/generate_labels.py \
  --limit 100 \
  --chunk-size 25 \
  --resume \
  --device cuda \
  --llm-profile llama32 \
  --backup-dir /content/drive/MyDrive/capstone_results

In [ ]:
# Quick check: how many labels do we have so far?
import json
path = "data/processed/prod_labels.json"
with open(path) as f:
    n = len(json.load(f)["records"])
print(f"Labeled prompts: {n}/100")
if n < 100:
    print("Re-run the previous cell until you see 100/100")
else:
    print("Ready for train + evaluate (next cell)")

In [ ]:
# Train LTR + PARS + evaluate (run this after labels are complete)
!python scripts/train_prod_m.py --target single --output checkpoints/ltr_pointwise.pt --device cuda
!python scripts/train_ranker.py --train-samples 100 --device cuda
!python scripts/evaluate.py --limit 100 --device cuda

!mkdir -p /content/drive/MyDrive/capstone_results
!cp -r checkpoints data/processed /content/drive/MyDrive/capstone_results/
print("final results saved to Drive")

## Results section (report figures)

Run after training finishes. Builds **main-paper style graphs**:
- length distributions (Fig. 2 style)
- latency vs request rate (Fig. 3 style)
- FCFS / LTR / OURS bar charts + % improvements

Figures + `results_section.md` are saved to Drive for your report.

In [ ]:
# RESULTS: tables + paper-style graphs → Drive
import os, glob
from IPython.display import display, Markdown, Image

FIG = "/content/drive/MyDrive/capstone_results/figures"
os.makedirs(FIG, exist_ok=True)

# Use how many labels you actually have (100 demo or up to 1000 live_run)
LIMIT = 100  # change to 200 / 500 / 1000 when ready

!pip install -q matplotlib
!python -u scripts/plot_results.py \
  --config configs/live_run.yaml \
  --limit {LIMIT} \
  --device cuda \
  --out-dir {FIG}

# Show report table
md_path = f"{FIG}/results_section.md"
if os.path.exists(md_path):
    display(Markdown(open(md_path).read()))

# Show every figure inline (copy these PNGs into the report)
print("\nFigures saved to:", FIG)
for path in sorted(glob.glob(f"{FIG}/fig_*.png")):
    print(path)
    display(Image(filename=path))

### If Colab disconnects during labeling

1. Re-run token + clone cells  
2. Re-run the **label generation** cell (it uses `--resume` and Drive backup)  
3. Continue until it says 100/100  
4. Then run the train + evaluate cell